In [0]:
%python
from pyspark.sql.window import Window
from pyspark.sql import functions as F

###bike_lakehouse.silver.crm_cust_info

In [0]:
crm_cust_info_df = spark.sql("""
                             with remove_dup as (
                             SELECT * ,
                                    row_number() over(PARTITION BY cst_id ORDER BY _load_timestamp DESC) AS rn
                             FROM bike_lakehouse.bronze.crm_cust_info
                             WHERE cst_id IS NOT NULL
                             ),
                             string_cleaning as (
                     
                             SELECT cst_id,cst_key,
                                    trim(cst_firstname) as cst_firstname,
                                    trim(cst_lastname) as cst_lastname,
                                    case
                                          when upper(trim(cst_marital_status)) = 'S' then 'single'
                                          when upper(trim(cst_marital_status)) ='M'  then 'married'
                                          else 'n/a'
                                    end as cst_marital_status, -- Normalize marital status values to readable format
                                    case
                                          when upper(trim(cst_gndr)) = 'F' then 'female'
                                          when upper(trim(cst_gndr)) = 'M' then 'male'
                                          else 'n/a'
                                    end as cst_gndr, -- Normalize gender values to readable format
                                    cast(cst_create_date as date),
                                    _load_timestamp,_source_system
                             FROM remove_dup
                             WHERE rn = 1
                             )

                             SELECT * FROM string_cleaning
                             """)


crm_cust_info_df.write.mode("overwrite").saveAsTable("bike_lakehouse.silver.crm_cust_info")

###bike_lakehouse.silver.crm_prd_info

In [0]:
crm_prd_info_df = spark.sql("""
                            
        SELECT
			prd_id,
			REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_') AS cat_id, -- Extract category ID
			SUBSTRING(prd_key, 7, LEN(prd_key)) AS prd_key,        -- Extract product key
			prd_nm,
			coalesce(prd_cost, 0) AS prd_cost,
			CASE 
				WHEN UPPER(TRIM(prd_line)) = 'M' THEN 'Mountain'
				WHEN UPPER(TRIM(prd_line)) = 'R' THEN 'Road'
				WHEN UPPER(TRIM(prd_line)) = 'S' THEN 'Other Sales'
				WHEN UPPER(TRIM(prd_line)) = 'T' THEN 'Touring'
				ELSE 'n/a'
			END AS prd_line, -- Map product line codes to descriptive values
            CAST(prd_start_dt AS DATE) AS prd_start_dt,
            date_sub(
                LEAD(cast(prd_start_dt as DATE)) OVER (PARTITION BY prd_key ORDER BY cast(prd_start_dt as DATE)),
                1
            ) AS prd_end_dt -- Calculate end date as one day before the next start date
		FROM bike_lakehouse.bronze.crm_prd_info
                            
                            
                            
                            """)


crm_prd_info_df.write.mode("overwrite").saveAsTable("bike_lakehouse.silver.crm_prd_info")

###bike_lakehouse.silver.crm_sales_details

In [0]:
crm_sales_details_df = spark.sql("""
        SELECT 
			sls_ord_num,
			sls_prd_key,
			sls_cust_id,
			CASE 
				WHEN sls_order_dt = 0 OR LEN(sls_order_dt) != 8 THEN NULL
				ELSE to_date(cast(sls_order_dt as STRING), 'yyyyMMdd')
			END AS sls_order_dt,
			CASE 
				WHEN sls_ship_dt = 0 OR LEN(sls_ship_dt) != 8 THEN NULL
				ELSE to_date(cast(sls_ship_dt as STRING),'yyyyMMdd')
			END AS sls_ship_dt,
			CASE 
				WHEN sls_due_dt = 0 OR LEN(sls_due_dt) != 8 THEN NULL
				ELSE to_date(cast(sls_due_dt as STRING),'yyyyMMdd')
			END AS sls_due_dt,
			CASE 
				WHEN sls_sales IS NULL OR sls_sales <= 0 OR sls_sales != sls_quantity * ABS(sls_price) 
					THEN sls_quantity * ABS(sls_price)
				ELSE sls_sales
			END AS sls_sales, -- Recalculate sales if original value is missing or incorrect
			sls_quantity,
			CASE 
				WHEN sls_price IS NULL OR sls_price <= 0 
					THEN sls_sales / NULLIF(sls_quantity, 0)
				ELSE sls_price  -- Derive price if original value is invalid
			END AS sls_price
		FROM bike_lakehouse.bronze.crm_sales_details
      

                                 
                                 """)


crm_sales_details_df.write.mode("overwrite").saveAsTable("bike_lakehouse.silver.crm_sales_details")

###bike_lakehouse.silver.erp_cust_az12

In [0]:
erp_cust_az12_df= spark.sql("""
        SELECT
			CASE
				WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid, 4, LEN(cid)) -- Remove 'NAS' prefix if present
				ELSE cid
			END AS cid, 
			CASE
				WHEN bdate > GETDATE() THEN NULL
				ELSE bdate
			END AS bdate, -- Set future birthdates to NULL
			CASE
				WHEN UPPER(TRIM(gen)) IN ('F', 'FEMALE') THEN 'Female'
				WHEN UPPER(TRIM(gen)) IN ('M', 'MALE') THEN 'Male'
				ELSE 'n/a'
			END AS gen -- Normalize gender values and handle unknown cases
		FROM bike_lakehouse.bronze.erp_cust_az12
                            """)


erp_cust_az12_df.write.mode("overwrite").saveAsTable("bike_lakehouse.silver.erp_cust_az12")

###bike_lakehouse.silver.erp_loc_a101

In [0]:
erp_loc_a101_df = spark.sql("""
        SELECT
			REPLACE(cid, '-', '') AS cid, 
			CASE
				WHEN TRIM(cntry) = 'DE' THEN 'Germany'
				WHEN TRIM(cntry) IN ('US', 'USA') THEN 'United States'
				WHEN TRIM(cntry) = '' OR cntry IS NULL THEN 'n/a'
				ELSE TRIM(cntry)
			END AS cntry -- Normalize and Handle missing or blank country codes
		FROM bike_lakehouse.bronze.erp_loc_a101
                            """)

erp_loc_a101_df.write.mode("overwrite").saveAsTable("bike_lakehouse.silver.erp_loc_a101")

###bike_lakehouse.silver.erp_px_cat_g1v2

In [0]:
erp_px_cat_g1v2_df = spark.sql("""
                               
        SELECT
			id,
			cat,
			subcat,
			maintenance
		FROM bike_lakehouse.bronze.erp_px_cat_g1v2
                               
                               """)


erp_px_cat_g1v2_df.write.mode("overwrite").saveAsTable("bike_lakehouse.silver.erp_px_cat_g1v2")